# LeetCode 1748 — Sum of Unique Elements bằng Python

Link đề: https://leetcode.com/problems/sum-of-unique-elements/

Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def sumOfUnique(self, nums: list[int]) -> int:

```

Mục tiêu học:

1. Hiểu đúng nghĩa **"unique" = xuất hiện đúng 1 lần** (khác với "các giá trị khác nhau" của `set`).
2. Đếm tần suất bằng `list.count`, `dict`, `collections.Counter`, và **mảng đếm** khi miền giá trị nhỏ.
3. Mẫu **hai pha**: pha 1 đếm, pha 2 lọc theo số lần đếm.
4. Mẫu **một pha có trạng thái**: cộng khi gặp lần 1, trừ khi gặp lần 2, bỏ qua từ lần 3.
5. So sánh `O(n²)` với `O(n)` và chọn cấu trúc dữ liệu phù hợp.
6. Tự giải biến thể: tích các phần tử unique, đếm phần tử unique, phần tử xuất hiện đúng k lần…

## 1. Bài toán yêu cầu gì?

Cho mảng số nguyên `nums`. Phần tử **unique** là phần tử xuất hiện **đúng một lần** trong mảng.

Trả về **tổng** các phần tử unique.

| nums | Phần tử unique | Kết quả |
|---|---|---:|
| `[1,2,3,2]` | `1, 3` | `4` |
| `[1,1,1,1,1]` | (không có) | `0` |
| `[1,2,3,4,5]` | `1,2,3,4,5` | `15` |

Constraints:

- `1 <= nums.length <= 100`
- `1 <= nums[i] <= 100`

Cẩn thận: số `2` trong ví dụ 1 xuất hiện 2 lần nên **không** được cộng — kể cả một lần.

In [1]:
nums = [1, 2, 3, 2]
for x in [1, 2, 3]:
    print(f"{x} xuất hiện {nums.count(x)} lần ->", "unique" if nums.count(x) == 1 else "KHÔNG unique")

1 xuất hiện 1 lần -> unique
2 xuất hiện 2 lần -> KHÔNG unique
3 xuất hiện 1 lần -> unique


## 2. Nền tảng Python cần biết

### 2.1. `list.count(x)`

Đếm số lần `x` xuất hiện trong list. Nó **duyệt toàn bộ list** mỗi lần gọi → tốn `O(n)`.

In [2]:
nums = [1, 2, 3, 2]
print(nums.count(2), nums.count(1), nums.count(9))

2 1 0


### 2.2. `set` — tập các giá trị khác nhau

`set(nums)` bỏ trùng lặp. Lưu ý: `set` cho ta "các giá trị **khác nhau**", **không phải** "các giá trị unique" theo nghĩa của đề.

In [3]:
nums = [1, 2, 3, 2]
print(set(nums), "-> tổng", sum(set(nums)), "(SAI với đề, đáp án là 4)")

{1, 2, 3} -> tổng 6 (SAI với đề, đáp án là 4)


### 2.3. Dictionary đếm tần suất

`freq[x] = freq.get(x, 0) + 1`: nếu `x` chưa có thì coi như 0, rồi cộng 1.

`freq.items()` trả các cặp `(key, value)` để duyệt.

In [4]:
freq = {}
for x in [1, 2, 3, 2]:
    freq[x] = freq.get(x, 0) + 1
print(freq)
for key, cnt in freq.items():
    print("giá trị", key, "xuất hiện", cnt, "lần")

{1: 1, 2: 2, 3: 1}
giá trị 1 xuất hiện 1 lần
giá trị 2 xuất hiện 2 lần
giá trị 3 xuất hiện 1 lần


### 2.4. `collections.Counter`

`Counter` là dict chuyên đếm. Truy cập key chưa có trả `0` (không `KeyError`).

In [5]:
from collections import Counter

c = Counter([1, 2, 3, 2])
print(c)
print(c[2], c[1], c[99])

Counter({2: 2, 1: 1, 3: 1})
2 1 0


### 2.5. Mảng đếm (counting array)

Vì `1 <= nums[i] <= 100`, ta dùng list 101 ô: `cnt[v]` = số lần giá trị `v` xuất hiện. Chỉ số chính là giá trị.

In [6]:
cnt = [0] * 101
for x in [1, 2, 3, 2]:
    cnt[x] += 1
print(cnt[:5])   # chỉ in 5 ô đầu: chỉ số 0..4

[0, 1, 2, 1, 0]


### 2.6. Generator expression có điều kiện

`sum(x for x in data if điều_kiện)` cộng các `x` thỏa điều kiện. Đọc là: "tổng của x, với x trong data, nếu điều kiện đúng".

In [7]:
data = [1, 2, 3, 4, 5]
print(sum(x for x in data if x % 2 == 1))   # 1 + 3 + 5

9


## 3. Cách trực giác — với mỗi phần tử, đếm xem nó xuất hiện mấy lần

Người mới hay nghĩ thẳng theo định nghĩa:

- Duyệt từng `x` trong `nums`.
- Nếu `nums.count(x) == 1` thì cộng `x` vào tổng.

In [8]:
def sum_of_unique_count(nums: list[int]) -> int:
    total = 0
    for x in nums:
        if nums.count(x) == 1:
            total += x
    return total

print(sum_of_unique_count([1, 2, 3, 2]))      # 4
print(sum_of_unique_count([1, 1, 1, 1, 1]))   # 0
print(sum_of_unique_count([1, 2, 3, 4, 5]))   # 15

4
0
15


### 3.1. Cách vét cạn hai vòng lặp (không dùng `count`)

Viết tường minh: với mỗi `i`, đếm số `j` có `nums[j] == nums[i]`. Đây chính là thứ `count` làm bên trong.

In [9]:
def sum_of_unique_two_loops(nums: list[int]) -> int:
    total = 0
    n = len(nums)
    for i in range(n):
        same = 0
        for j in range(n):
            if nums[j] == nums[i]:
                same += 1
        if same == 1:
            total += nums[i]
    return total

print(sum_of_unique_two_loops([1, 2, 3, 2]))

4


## 4. Nút thắt và insight

**Nút thắt:** mỗi lần gọi `nums.count(x)` lại duyệt cả mảng → tổng `n × n` phép so sánh. Với `[2, 2, 2, …]`, ta đếm đi đếm lại số `2` cả trăm lần — hoàn toàn lặp lại.

**Insight:** "số lần xuất hiện" của mỗi giá trị là thông tin **chung**, chỉ cần tính **một lần** rồi tra lại. Đó là **bảng tần suất**:

1. **Pha 1 — đếm:** duyệt mảng một lần, ghi `freq[x]`.
2. **Pha 2 — lọc:** cộng các giá trị có `freq == 1`.

Tổng chi phí: `n` (đếm) + số giá trị khác nhau (lọc) → `O(n)`.

In [10]:
# Đếm số phép so sánh của cách count so với cách đếm một lần
def comparisons_count_method(nums):
    return len(nums) * len(nums)

for n in [4, 10, 100]:
    print(f"n={n:>3}: cách count ~{comparisons_count_method([0]*n):>5} phép so sánh | bảng tần suất ~{n} lần cập nhật")

n=  4: cách count ~   16 phép so sánh | bảng tần suất ~4 lần cập nhật
n= 10: cách count ~  100 phép so sánh | bảng tần suất ~10 lần cập nhật
n=100: cách count ~10000 phép so sánh | bảng tần suất ~100 lần cập nhật


## 5. Lời giải chính — đếm tần suất rồi lọc

### 5.1. Pha 1: xây bảng tần suất

In [11]:
nums = [4, 1, 4, 7, 1, 9]
freq = {}
for x in nums:
    freq[x] = freq.get(x, 0) + 1
print("freq =", freq)

freq = {4: 2, 1: 2, 7: 1, 9: 1}


### 5.2. Pha 2: cộng các giá trị xuất hiện đúng 1 lần

Chú ý: ta cộng **key** (giá trị), mỗi key một lần — vì nó chỉ xuất hiện 1 lần nên cộng 1 lần là đúng.

In [12]:
total = 0
for value, cnt in freq.items():
    if cnt == 1:
        total += value
print("tổng unique =", total)   # 7 + 9 = 16

tổng unique = 16


### 5.3. Ghép thành hàm

In [13]:
def sum_of_unique(nums: list[int]) -> int:
    freq = {}
    for x in nums:                       # pha 1: đếm
        freq[x] = freq.get(x, 0) + 1
    total = 0
    for value, cnt in freq.items():      # pha 2: lọc
        if cnt == 1:
            total += value
    return total

print(sum_of_unique([1, 2, 3, 2]), sum_of_unique([1, 1, 1, 1, 1]), sum_of_unique([1, 2, 3, 4, 5]))

4 0 15


## 6. TRACE từng bước

Bảng trace pha 1 cho `nums = [1, 2, 3, 2]`:

| i | x | freq trước | freq sau |
|---:|---:|---|---|
| 0 | 1 | `{}` | `{1:1}` |
| 1 | 2 | `{1:1}` | `{1:1, 2:1}` |
| 2 | 3 | `{1:1, 2:1}` | `{1:1, 2:1, 3:1}` |
| 3 | 2 | `{1:1, 2:1, 3:1}` | `{1:1, 2:2, 3:1}` |

Pha 2:

| value | cnt | cnt == 1? | total |
|---:|---:|---|---:|
| 1 | 1 | Có | 1 |
| 2 | 2 | Không | 1 |
| 3 | 1 | Có | 4 |

In [14]:
def trace_sum_of_unique(nums: list[int]) -> int:
    print("PHA 1 — đếm")
    freq = {}
    for i, x in enumerate(nums):
        before = dict(freq)
        freq[x] = freq.get(x, 0) + 1
        print(f"  i={i}, x={x}: {before} -> {freq}")
    print("PHA 2 — lọc")
    total = 0
    for value, cnt in freq.items():
        ok = cnt == 1
        if ok:
            total += value
        print(f"  value={value}, cnt={cnt}, unique? {ok} -> total={total}")
    print("=> trả về", total)
    return total

trace_sum_of_unique([1, 2, 3, 2])

PHA 1 — đếm
  i=0, x=1: {} -> {1: 1}
  i=1, x=2: {1: 1} -> {1: 1, 2: 1}
  i=2, x=3: {1: 1, 2: 1} -> {1: 1, 2: 1, 3: 1}
  i=3, x=2: {1: 1, 2: 1, 3: 1} -> {1: 1, 2: 2, 3: 1}
PHA 2 — lọc
  value=1, cnt=1, unique? True -> total=1
  value=2, cnt=2, unique? False -> total=1
  value=3, cnt=1, unique? True -> total=4
=> trả về 4


4

`dict(freq)` tạo **bản sao** của dict tại thời điểm đó, để in "trước" không bị thay đổi theo "sau".

### 6.1. Trace trường hợp không có phần tử unique

In [15]:
trace_sum_of_unique([1, 1, 1, 1, 1])

PHA 1 — đếm
  i=0, x=1: {} -> {1: 1}
  i=1, x=1: {1: 1} -> {1: 2}
  i=2, x=1: {1: 2} -> {1: 3}
  i=3, x=1: {1: 3} -> {1: 4}
  i=4, x=1: {1: 4} -> {1: 5}
PHA 2 — lọc
  value=1, cnt=5, unique? False -> total=0
=> trả về 0


0

## 7. Code nộp LeetCode

In [16]:
from collections import Counter

class Solution:
    def sumOfUnique(self, nums: list[int]) -> int:
        freq = Counter(nums)
        return sum(value for value, cnt in freq.items() if cnt == 1)

sol = Solution()
print(sol.sumOfUnique([1, 2, 3, 2]), sol.sumOfUnique([1, 1, 1, 1, 1]), sol.sumOfUnique([1, 2, 3, 4, 5]))

4 0 15


## 8. Các lời giải khác

### 8.1. Mảng đếm 101 ô (tận dụng `1 <= nums[i] <= 100`)

Không cần băm; truy cập chỉ số trực tiếp. Pha 2 duyệt 100 ô cố định.

In [17]:
def sum_of_unique_array(nums: list[int]) -> int:
    cnt = [0] * 101
    for x in nums:
        cnt[x] += 1
    return sum(v for v in range(1, 101) if cnt[v] == 1)

print(sum_of_unique_array([1, 2, 3, 2]), sum_of_unique_array([1, 1, 1, 1, 1]))

4 0


In [18]:
class SolutionArray:
    def sumOfUnique(self, nums: list[int]) -> int:
        cnt = [0] * 101
        for x in nums:
            cnt[x] += 1
        return sum(v for v in range(1, 101) if cnt[v] == 1)

print(SolutionArray().sumOfUnique([1, 2, 3, 4, 5]))

15


### 8.2. Một pha có trạng thái: cộng lần 1, trừ lần 2, bỏ qua từ lần 3

Ý tưởng sáng tạo: khi duyệt, ta có thể **cập nhật đáp án ngay**:

- Gặp `x` **lần đầu**: tạm coi nó unique → `total += x`.
- Gặp `x` **lần thứ hai**: hóa ra không unique → **hoàn tác**: `total -= x`.
- Gặp `x` **lần thứ ba trở đi**: đã hoàn tác rồi, không làm gì.

Chỉ cần một vòng lặp, không cần pha lọc.

In [19]:
def sum_of_unique_one_pass(nums: list[int]) -> int:
    seen = {}            # x -> số lần đã gặp
    total = 0
    for x in nums:
        c = seen.get(x, 0)
        if c == 0:
            total += x
        elif c == 1:
            total -= x
        seen[x] = c + 1
    return total

print(sum_of_unique_one_pass([1, 2, 3, 2]), sum_of_unique_one_pass([1, 1, 1, 1, 1]), sum_of_unique_one_pass([5, 5, 5, 6]))

4 0 6


In [20]:
def trace_one_pass(nums):
    seen, total = {}, 0
    for x in nums:
        c = seen.get(x, 0)
        action = "cộng" if c == 0 else ("TRỪ (hoàn tác)" if c == 1 else "bỏ qua")
        if c == 0:
            total += x
        elif c == 1:
            total -= x
        seen[x] = c + 1
        print(f"x={x}: đã gặp {c} lần -> {action:<15} total={total}")
    return total

trace_one_pass([2, 7, 2, 2, 5])

x=2: đã gặp 0 lần -> cộng            total=2
x=7: đã gặp 0 lần -> cộng            total=9
x=2: đã gặp 1 lần -> TRỪ (hoàn tác)  total=7
x=2: đã gặp 2 lần -> bỏ qua          total=7
x=5: đã gặp 0 lần -> cộng            total=12


12

In [21]:
class SolutionOnePass:
    def sumOfUnique(self, nums: list[int]) -> int:
        seen = {}
        total = 0
        for x in nums:
            c = seen.get(x, 0)
            if c == 0:
                total += x
            elif c == 1:
                total -= x
            seen[x] = c + 1
        return total

print(SolutionOnePass().sumOfUnique([1, 2, 3, 2]))

4


### 8.3. Hai `set`: "đã gặp" và "bị trùng"

`seen` chứa mọi giá trị đã gặp, `dup` chứa giá trị gặp ≥ 2 lần. Đáp án = `sum(seen - dup)`.

`a - b` với hai `set` là **hiệu tập hợp**: phần tử thuộc `a` mà không thuộc `b`.

In [22]:
print({1, 2, 3} - {2})

def sum_of_unique_sets(nums: list[int]) -> int:
    seen, dup = set(), set()
    for x in nums:
        if x in seen:
            dup.add(x)
        seen.add(x)
    return sum(seen - dup)

print(sum_of_unique_sets([1, 2, 3, 2]), sum_of_unique_sets([1, 1, 1, 1, 1]))

{1, 3}
4 0


### 8.4. Bảng so sánh

Gọi `n = len(nums)`, `U` = số giá trị khác nhau (≤ 100), `V` = độ rộng miền giá trị (100).

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---:|---:|---|
| `nums.count(x)` | đếm lại cho từng phần tử | `O(n²)` | `O(1)` | Theo sát định nghĩa, chậm |
| Hai vòng lặp | như trên, viết tay | `O(n²)` | `O(1)` | Để hiểu `count` |
| dict / `Counter` hai pha | đếm rồi lọc | `O(n)` trung bình | `O(U)` | **Lời giải chính** |
| Mảng 101 ô | đếm theo chỉ số | `O(n + V)` | `O(V)` | Dùng được vì miền nhỏ |
| Một pha cộng/trừ | hoàn tác khi gặp lần 2 | `O(n)` | `O(U)` | Sáng tạo, gọn |
| Hai `set` | `seen - dup` | `O(n)` | `O(U)` | Không cần đếm số |

## 9. Vì sao lời giải đúng?

**Cách hai pha.** Sau pha 1, `freq[v]` đúng bằng số lần `v` xuất hiện (mỗi phần tử làm tăng đúng ô của nó một lần). Pha 2 cộng `v` **đúng một lần** khi và chỉ khi `freq[v] == 1` — chính là định nghĩa unique, và một giá trị unique đóng góp đúng `v` vào tổng.

**Cách một pha — bất biến:** sau khi duyệt `k` phần tử đầu, `total` = tổng các giá trị xuất hiện **đúng một lần trong `k` phần tử đầu**.

- Gặp `x` lần 1: `x` vừa thành unique (trong tiền tố) → cộng. Đúng.
- Gặp `x` lần 2: `x` vừa hết unique → trừ đúng phần đã cộng. Đúng.
- Gặp lần ≥ 3: `x` vẫn không unique, đã không nằm trong `total` → giữ. Đúng.

Hết mảng, tiền tố là cả mảng → `total` là đáp án.

In [23]:
# Kiểm chứng bất biến của cách một pha trên từng tiền tố
nums = [3, 1, 3, 4, 3, 1, 8]
seen, total = {}, 0
for k, x in enumerate(nums, start=1):
    c = seen.get(x, 0)
    if c == 0: total += x
    elif c == 1: total -= x
    seen[x] = c + 1
    assert total == sum_of_unique_count(nums[:k])
    print(f"tiền tố {nums[:k]} -> total = {total}")
print("Bất biến đúng ở mọi bước")

tiền tố [3] -> total = 3
tiền tố [3, 1] -> total = 4
tiền tố [3, 1, 3] -> total = 1
tiền tố [3, 1, 3, 4] -> total = 5
tiền tố [3, 1, 3, 4, 3] -> total = 5
tiền tố [3, 1, 3, 4, 3, 1] -> total = 4
tiền tố [3, 1, 3, 4, 3, 1, 8] -> total = 12
Bất biến đúng ở mọi bước


## 10. Độ phức tạp

Gọi `n = len(nums)`, `U` = số giá trị khác nhau.

- **Hai pha với `dict`/`Counter`:** pha 1 `O(n)` (mỗi thao tác dict `O(1)` trung bình), pha 2 `O(U)` với `U ≤ n` → **Time `O(n)`**, **Space `O(U)`**.
- **Mảng 101 ô:** **Time `O(n + 100)`**, **Space `O(101)`** — hằng số, có thể coi `O(1)` vì miền giá trị cố định.
- **`count`:** **Time `O(n²)`**, Space `O(1)`.

Với `n ≤ 100`, `O(n²)` = 10.000 phép so sánh vẫn qua, nhưng tư duy "đếm một lần" mới mở rộng được khi `n` lớn.

## 11. Bộ test

Nhóm test:

1. Ba ví dụ của đề.
2. Một phần tử.
3. Mọi phần tử trùng / mọi phần tử khác nhau.
4. Giá trị xuất hiện 3 lần (để bắt lỗi "trừ nhiều lần" ở cách một pha).
5. Biên giá trị 1 và 100.
6. Đối chiếu mọi cách trên dữ liệu ngẫu nhiên.

In [24]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS (={actual})")

sol = Solution()
check("đề 1", sol.sumOfUnique([1, 2, 3, 2]), 4)
check("đề 2", sol.sumOfUnique([1, 1, 1, 1, 1]), 0)
check("đề 3", sol.sumOfUnique([1, 2, 3, 4, 5]), 15)
check("một phần tử", sol.sumOfUnique([42]), 42)
check("xuất hiện 3 lần", sol.sumOfUnique([7, 7, 7, 1]), 1)
check("hai cặp trùng", sol.sumOfUnique([5, 5, 6, 6]), 0)
check("biên 1 và 100", sol.sumOfUnique([1, 100, 100]), 1)
check("100 giá trị khác nhau", sol.sumOfUnique(list(range(1, 101))), 5050)

đề 1                         PASS (=4)
đề 2                         PASS (=0)
đề 3                         PASS (=15)
một phần tử                  PASS (=42)
xuất hiện 3 lần              PASS (=1)
hai cặp trùng                PASS (=0)
biên 1 và 100                PASS (=1)
100 giá trị khác nhau        PASS (=5050)


In [25]:
import random

rng = random.Random(1748)
impls = [sum_of_unique_count, sum_of_unique_two_loops, sum_of_unique, sum_of_unique_array,
         sum_of_unique_one_pass, sum_of_unique_sets,
         lambda a: Solution().sumOfUnique(a), lambda a: SolutionArray().sumOfUnique(a),
         lambda a: SolutionOnePass().sumOfUnique(a)]
for _ in range(500):
    n = rng.randint(1, 100)
    hi = rng.choice([3, 10, 100])          # miền nhỏ -> nhiều trùng lặp
    nums = [rng.randint(1, hi) for _ in range(n)]
    outs = {f(nums) for f in impls}
    assert len(outs) == 1, (nums, outs)
print("500 mảng ngẫu nhiên: 9 cách cho cùng kết quả")

500 mảng ngẫu nhiên: 9 cách cho cùng kết quả


## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `sum(set(nums))` — nhầm "khác nhau" với "unique"

In [26]:
nums = [1, 2, 3, 2]
print("sai:", sum(set(nums)), "| đúng:", sum_of_unique(nums))

sai: 6 | đúng: 4


**Sửa:** phải đếm số lần xuất hiện; chỉ lấy giá trị có tần suất **bằng 1**.

### Lỗi 2: Sai điều kiện lọc

Ví dụ nghĩ "trùng là xuất hiện đúng 2 lần" nên viết `cnt != 2`, hoặc viết `cnt >= 1`. Giá trị xuất hiện 3 lần sẽ bị cộng nhầm. Điều kiện duy nhất đúng là `cnt == 1`.

In [27]:
def wrong_condition(nums):
    freq = Counter(nums)
    return sum(v for v, c in freq.items() if c != 2)   # tưởng "trùng" là đúng 2 lần

print("sai:", wrong_condition([7, 7, 7, 1]), "| đúng:", sum_of_unique([7, 7, 7, 1]))

sai: 8 | đúng: 1


### Lỗi 3: Cách một pha nhưng trừ **mỗi lần** gặp lại

Nếu viết `else: total -= x` cho mọi lần gặp sau, giá trị xuất hiện 3 lần bị trừ 2 lần → âm.

In [28]:
def wrong_one_pass(nums):
    seen, total = set(), 0
    for x in nums:
        if x not in seen:
            total += x
            seen.add(x)
        else:
            total -= x          # trừ cả ở lần 3, 4, ...
    return total

print("sai:", wrong_one_pass([7, 7, 7, 1]), "| đúng:", sum_of_unique_one_pass([7, 7, 7, 1]))

sai: -6 | đúng: 1


**Sửa:** chỉ trừ ở **lần thứ hai** (cần đếm, hoặc dùng thêm set `dup`).

### Lỗi 4: Xóa phần tử khỏi list trong lúc đang duyệt nó

`for x in nums: if ...: nums.remove(x)` làm vòng lặp **nhảy cóc** phần tử, kết quả khó đoán. **Sửa:** không sửa list đang duyệt; dùng bảng tần suất.

In [29]:
nums = [2, 2, 3]
for x in nums:
    if nums.count(x) > 1:
        nums.remove(x)
print("list sau khi 'xóa trùng' lúc đang duyệt:", nums, "<- số 2 (vốn bị trùng) vẫn còn, vì vòng lặp đã nhảy cóc")
print("tổng sai:", sum(nums), "| đúng:", sum_of_unique([2, 2, 3]))

list sau khi 'xóa trùng' lúc đang duyệt: [2, 3] <- số 2 (vốn bị trùng) vẫn còn, vì vòng lặp đã nhảy cóc
tổng sai: 5 | đúng: 3


## 13. Mẹo nhận dạng pattern

- Đề nói **"xuất hiện đúng một lần / đúng k lần / nhiều nhất / ít nhất"** → **bảng tần suất** (`Counter`).
- Phân biệt: **"distinct" (khác nhau)** → `set`; **"unique" (đúng 1 lần)** → đếm.
- Miền giá trị nhỏ và biết trước (`1..100`) → **mảng đếm**.
- Có thể cập nhật đáp án **ngay khi đếm** nếu biết "hoàn tác" khi trạng thái thay đổi (lần 1 → lần 2).

Bài cùng họ: 387 First Unique Character in a String, 136 Single Number, 1207 Unique Number of Occurrences, 2085 Count Common Words With One Occurrence.

## 14. Biến thể

### 14.1. **Đếm** số phần tử unique

In [30]:
def count_unique(nums):
    return sum(1 for c in Counter(nums).values() if c == 1)

print(count_unique([1, 2, 3, 2]))   # 2

2


### 14.2. Tổng các giá trị xuất hiện **đúng k lần**

Tổng quát hóa: thay `== 1` bằng `== k`. Ở đây ta cộng **mỗi giá trị một lần** (không nhân k).

In [31]:
def sum_values_with_freq(nums, k):
    return sum(v for v, c in Counter(nums).items() if c == k)

print(sum_values_with_freq([1, 2, 2, 3, 3, 4, 4, 4], 2))   # 2 + 3 = 5
print(sum_values_with_freq([1, 2, 2, 3, 3, 4, 4, 4], 3))   # 4

5
4


### 14.3. Bài 387 — Vị trí ký tự unique đầu tiên trong chuỗi

Cùng khung "đếm trước, rồi duyệt lại theo thứ tự gốc".

In [32]:
def first_uniq_char(s):
    freq = Counter(s)
    for i, ch in enumerate(s):
        if freq[ch] == 1:
            return i
    return -1

print(first_uniq_char("leetcode"), first_uniq_char("loveleetcode"), first_uniq_char("aabb"))

0 2 -1


## 15. Bài tập tự luyện

Các hàm TODO không được gọi khi chạy notebook. Đáp án ở mục 17.

### Bài 1 — Tính tay

`nums = [4, 5, 7, 5, 4, 8]`. Lập bảng tần suất và tính tổng unique.

### Bài 2 — Dự đoán output

Với `nums = [3, 3, 9]`: `sum(set(nums))`, `Counter(nums)[3]`, `Counter(nums)[4]` lần lượt ra gì?

### Bài 3 — Điền TODO: hai pha với dict

In [33]:
def exercise_sum_of_unique(nums: list[int]) -> int:
    # TODO 1: freq = {} và đếm bằng freq.get(x, 0) + 1
    # TODO 2: cộng các value có cnt == 1
    pass

# assert exercise_sum_of_unique([1, 2, 3, 2]) == 4
# assert exercise_sum_of_unique([1, 1, 1, 1, 1]) == 0

### Bài 4 — Tự viết cách một pha cộng/trừ

In [34]:
def exercise_one_pass(nums: list[int]) -> int:
    # TODO: lần 1 cộng, lần 2 trừ, lần >= 3 bỏ qua
    pass

# assert exercise_one_pass([7, 7, 7, 1]) == 1

### Bài 5 — Biến thể: tích các phần tử unique

Trả về tích các giá trị xuất hiện đúng 1 lần. Không có phần tử unique thì trả `1` (tích rỗng).

In [35]:
def exercise_product_of_unique(nums: list[int]) -> int:
    # TODO: giống tổng nhưng khởi tạo 1 và nhân
    pass

# assert exercise_product_of_unique([2, 3, 3, 5]) == 10
# assert exercise_product_of_unique([1, 1]) == 1

### Bài 6 — Mini challenge: các giá trị unique theo **thứ tự xuất hiện**, chỉ dùng một lần đếm

Trả list các phần tử unique giữ nguyên thứ tự trong `nums`.

In [36]:
def exercise_unique_in_order(nums: list[int]) -> list[int]:
    # TODO: Counter trước, rồi duyệt nums theo thứ tự gốc
    pass

# assert exercise_unique_in_order([4, 5, 7, 5, 4, 8]) == [7, 8]
# assert exercise_unique_in_order([1, 1]) == []

## 16. Gợi ý 3 mức

**Bài 1** — M1: đếm từng giá trị. M2: `{4:2, 5:2, 7:1, 8:1}`. M3: cộng các giá trị có tần suất 1.

**Bài 2** — M1: `set` bỏ trùng. M2: `Counter` trả 0 với key không có. M3: `12`, `2`, `0`.

**Bài 3** — M1: hai vòng lặp tách biệt. M2: `for value, cnt in freq.items()`. M3: `if cnt == 1: total += value`.

**Bài 4** — M1: cần biết "đã gặp mấy lần" → dict. M2: `c = seen.get(x, 0)`. M3: `if c == 0: +x; elif c == 1: -x; seen[x] = c + 1`.

**Bài 5** — M1: phần tử trung hòa của phép nhân là 1. M2: dùng `Counter`. M3: `for v, c in freq.items(): if c == 1: prod *= v`.

**Bài 6** — M1: dict không cho biết thứ tự gốc của các lần xuất hiện ⇒ duyệt lại `nums`. M2: `freq = Counter(nums)`. M3: `[x for x in nums if freq[x] == 1]`.

## 17. Đáp án tham khảo

**Bài 1:** tần suất `{4:2, 5:2, 7:1, 8:1}` → tổng unique `7 + 8 = 15`.

**Bài 2:** `12`, `2`, `0`.

In [37]:
print(Counter([4, 5, 7, 5, 4, 8]), sum_of_unique([4, 5, 7, 5, 4, 8]))
print(sum(set([3, 3, 9])), Counter([3, 3, 9])[3], Counter([3, 3, 9])[4])

Counter({4: 2, 5: 2, 7: 1, 8: 1}) 15
12 2 0


In [38]:
def answer_sum_of_unique(nums):
    freq = {}
    for x in nums:
        freq[x] = freq.get(x, 0) + 1
    total = 0
    for value, cnt in freq.items():
        if cnt == 1:
            total += value
    return total


def answer_one_pass(nums):
    seen, total = {}, 0
    for x in nums:
        c = seen.get(x, 0)
        if c == 0:
            total += x
        elif c == 1:
            total -= x
        seen[x] = c + 1
    return total


def answer_product_of_unique(nums):
    prod = 1
    for v, c in Counter(nums).items():
        if c == 1:
            prod *= v
    return prod


def answer_unique_in_order(nums):
    freq = Counter(nums)
    return [x for x in nums if freq[x] == 1]


assert answer_sum_of_unique([1, 2, 3, 2]) == 4
assert answer_sum_of_unique([1, 1, 1, 1, 1]) == 0
assert answer_one_pass([7, 7, 7, 1]) == 1
assert answer_product_of_unique([2, 3, 3, 5]) == 10
assert answer_product_of_unique([1, 1]) == 1
assert answer_unique_in_order([4, 5, 7, 5, 4, 8]) == [7, 8]
assert answer_unique_in_order([1, 1]) == []
rng = random.Random(48)
for _ in range(300):
    nums = [rng.randint(1, 8) for _ in range(rng.randint(1, 15))]
    assert answer_one_pass(nums) == answer_sum_of_unique(nums) == sum(answer_unique_in_order(nums))
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
from collections import Counter

class Solution:
    def sumOfUnique(self, nums: list[int]) -> int:
        freq = Counter(nums)                         # pha 1: đếm
        return sum(v for v, c in freq.items() if c == 1)   # pha 2: lọc
```

Template "đếm rồi lọc":

```text
freq = Counter(data)
kết_quả = gộp(v for v, c in freq.items() if điều_kiện(c))
```

Template "một pha có hoàn tác":

```text
c = seen.get(x, 0)
lần 1 -> áp dụng; lần 2 -> hoàn tác; lần >= 3 -> bỏ qua
seen[x] = c + 1
```

Checklist trước khi submit:

- [ ] "Unique" = xuất hiện **đúng 1 lần**, không phải `set`?
- [ ] Điều kiện là `== 1` (không phải `!= 2`, `<= 1`)?
- [ ] Mỗi giá trị unique chỉ được cộng **một lần**?
- [ ] Nếu dùng cách một pha: chỉ trừ ở lần gặp thứ **hai**?
- [ ] Đã thử mảng toàn trùng (đáp án 0) và giá trị xuất hiện 3 lần?